In [1]:
import pandas as pd
import numpy as np
import sqlite3
import random
from datetime import datetime, timedelta

np.random.seed(42)

n_rows = 5000

asset_types = ['Scanner', 'scanner', 'SCANNER', 'Laptop', 'laptop', 'Conveyor Sensor', 'Router', 'Switch', None]
statuses = ['Active', 'Active', 'Active', 'Failed', 'Maintenance', 'Offline', 'Broken', None]
locations = ['Zone A', 'Zone B', 'Zone C', 'Zone D', 'Loading Dock', 'zone_a', 'ZONE B']

raw_data = {
    'Asset_ID': [f"AST-{random.randint(1000, 9999)}" for _ in range(n_rows)],
    'Asset_Type': np.random.choice(asset_types, n_rows),
    'Location': np.random.choice(locations, n_rows),
    'Deployment_Date': [
        (datetime(2020, 1, 1) + timedelta(days=random.randint(0, 2000))).strftime('%Y-%m-%d') 
        if random.random() > 0.1 else '1900-01-01' for _ in range(n_rows)
    ],
    'Failure_Count': np.random.normal(loc=2, scale=3, size=n_rows).astype(int),
    'Current_Status': np.random.choice(statuses, n_rows)
}
df = pd.DataFrame(raw_data)

for col in ['Failure_Count', 'Deployment_Date']:
    df.loc[df.sample(frac=0.05).index, col] = np.nan



df['Location'] = df['Location'].str.replace('_', ' ').str.title()

df['Asset_Type'] = df['Asset_Type'].str.title().fillna('Unknown')
df['Current_Status'] = df['Current_Status'].str.title().fillna('Unknown')
df['Failure_Count'] = df['Failure_Count'].fillna(0).clip(lower=0).astype(int)
df['Deployment_Date'] = pd.to_datetime(df['Deployment_Date'], errors='coerce')
df.loc[df['Deployment_Date'] == '1900-01-01', 'Deployment_Date'] = pd.NaT


df.to_csv('cleaned_asset_data.csv', index=False)


conn = sqlite3.connect('ecommerce_assets.db')
df.to_sql('asset_operations', conn, if_exists='replace', index=False)

query_locations = """
SELECT 
    Location, 
    COUNT(Asset_ID) as Total_Assets,
    SUM(CASE WHEN Current_Status IN ('Broken', 'Failed', 'Offline') THEN 1 ELSE 0 END) as Down_Assets,
    SUM(Failure_Count) as Total_Failures
FROM asset_operations
GROUP BY Location
ORDER BY Down_Assets DESC;
"""
df_locations = pd.read_sql_query(query_locations, conn)
print("--- WAREHOUSE ZONE BOTTLENECKS (REPAIRED) ---")
print(df_locations.to_string(index=False))

query_hardware = """
SELECT 
    Asset_Type,
    COUNT(Asset_ID) as Total_Units,
    AVG(Failure_Count) as Avg_Failures_Per_Unit
FROM asset_operations
WHERE Asset_Type != 'Unknown'
GROUP BY Asset_Type
ORDER BY Avg_Failures_Per_Unit DESC;
"""
df_hardware = pd.read_sql_query(query_hardware, conn)
print("--- HARDWARE DEFECT REPORT ---")
print(df_hardware.to_string(index=False))

conn.close()

--- WAREHOUSE ZONE BOTTLENECKS (REPAIRED) ---
    Location  Total_Assets  Down_Assets  Total_Failures
      Zone B          1521          585            2938
      Zone A          1433          544            2790
      Zone C           665          245            1307
Loading Dock           704          241            1341
      Zone D           677          240            1489
--- HARDWARE DEFECT REPORT ---
     Asset_Type  Total_Units  Avg_Failures_Per_Unit
Conveyor Sensor          581               2.077453
        Scanner         1671               1.991023
         Laptop         1107               1.949413
         Router          540               1.927778
         Switch          559               1.837209
